# Tutorial: Threads in C for Parallel Computing

> **Audience**: Students learning parallel and distributed computing
>
> **Scope**: POSIX threads (pthreads), race conditions, mutexes, semaphores, and atomic operations
>
> **Note**: This notebook is written as a *Jupyter-style tutorial*. Code cells are standard C programs that can be compiled and run from a terminal.

---

## 1. Why Threads?

A **thread** is a lightweight unit of execution within a process. Unlike processes, threads:
- Share the same address space
- Share global variables and heap memory
- Have lower creation and context-switch overhead

### Why use threads in parallel computing?
- Exploit **multi-core CPUs**
- Overlap computation and I/O
- Decompose a problem into independent tasks

---

## 2. POSIX Threads (pthreads)

The POSIX thread (pthread) library is the standard threading API in C/C++ on Unix-like systems.

### Key concepts
- Thread creation and termination
- Passing arguments to threads
- Synchronization and mutual exclusion

### Compilation
gcc program.c -o program -pthread

## 3. Creating a Thread

In [1]:
%%writefile thread_1.c

#include <stdio.h>
#include <pthread.h>

void* thread_func(void* arg) {
    printf("Hello from thread!\n");
    return NULL;
}

int main() {
    pthread_t t;

    pthread_create(&t, NULL, thread_func, NULL);
    pthread_join(t, NULL);

    printf("Hello from main thread!\n");
    return 0;
}

Writing thread_1.c


In [ ]:
!gcc thread_1.c -o thread_1
!./thread_1

Hello from thread!
Hello from main thread!


## 4. Passing Arguments to Threads

In [13]:
%%writefile thread_2.c

#include <stdio.h>
#include <pthread.h>

void* print_id(void* arg) {
    int id = *(int*)arg;
    printf("Thread ID: %d\n", id);
    return NULL;
}

int main() {
    pthread_t threads[4];
    int ids[4];

    for (int i = 0; i < 4; i++) {
        ids[i] = i;
        pthread_create(&threads[i], NULL, print_id, &ids[i]);
    }

    for (int i = 0; i < 4; i++) {
        pthread_join(threads[i], NULL);
    }
    return 0;
}


Writing thread_2.c


In [16]:
!gcc thread_2.c -o thread_2
!./thread_2

Thread ID: 0
Thread ID: 1
Thread ID: 2
Thread ID: 3


## 5. Data Sharing Between Threads

Threads share:
- Global variables
- Heap memory

```c
int shared_counter = 0;
```

This is powerful—but dangerous.

---
## 6. Race Conditions

A **race condition** occurs when:
1. Two or more threads access shared data
2. At least one thread modifies it
3. The final result depends on execution order

### Example: Race Condition

In [15]:
%%writefile thread_3.c

#include <stdio.h>
#include <pthread.h>

int counter = 0;

void* increment(void* arg) {
    for (int i = 0; i < 1000000; i++) {
        counter++;
    }
    return NULL;
}

int main() {
    pthread_t t1, t2;

    pthread_create(&t1, NULL, increment, NULL);
    pthread_create(&t2, NULL, increment, NULL);

    pthread_join(t1, NULL);
    pthread_join(t2, NULL);

    printf("Final counter: %d\n", counter);
    return 0;
}

Writing thread_3.c


In [20]:
!gcc thread_3.c -o thread_3
!./thread_3

Final counter: 1733830


### Expected vs Actual
- Expected: `2000000`
- Actual: **Often less** ❌

Why?

```text
counter++  ≡  load → add → store
```

These steps are **not atomic**.

---

## 7. Mutexes (Locks)

A **mutex** ensures mutual exclusion: only one thread enters a critical section at a time.

### Critical Section
```text
pthread_mutex_lock()
  counter++
pthread_mutex_unlock()
```

### Using pthread_mutex_t

In [21]:
%%writefile thread_4.c

#include <stdio.h>
#include <pthread.h>

int counter = 0;
pthread_mutex_t lock;

void* increment(void* arg) {
    for (int i = 0; i < 1000000; i++) {
        pthread_mutex_lock(&lock);
        counter++;
        pthread_mutex_unlock(&lock);
    }
    return NULL;
}

int main() {
    pthread_t t1, t2;
    pthread_mutex_init(&lock, NULL);

    pthread_create(&t1, NULL, increment, NULL);
    pthread_create(&t2, NULL, increment, NULL);

    pthread_join(t1, NULL);
    pthread_join(t2, NULL);

    pthread_mutex_destroy(&lock);
    printf("Final counter: %d\n", counter);
    return 0;
}

Writing thread_4.c


In [22]:
!gcc thread_4.c -o thread_4
!./thread_4

Final counter: 2000000


## 8. Deadlocks (Conceptual)

A **deadlock** occurs when:
- Thread A holds Lock 1 and waits for Lock 2
- Thread B holds Lock 2 and waits for Lock 1

Avoid by:
- Lock ordering
- Minimizing lock scope

---

In [23]:
%%writefile thread_5.c

#include <stdio.h>
#include <pthread.h>
#include <unistd.h>

pthread_mutex_t lock1;
pthread_mutex_t lock2;

void* thread1(void* arg) {
    pthread_mutex_lock(&lock1);
    printf("Thread 1 acquired lock1\n");

    sleep(1);  // Force interleaving

    printf("Thread 1 waiting for lock2\n");
    pthread_mutex_lock(&lock2);   // DEADLOCK here

    printf("Thread 1 acquired lock2\n");

    pthread_mutex_unlock(&lock2);
    pthread_mutex_unlock(&lock1);
    return NULL;
}

void* thread2(void* arg) {
    pthread_mutex_lock(&lock2);
    printf("Thread 2 acquired lock2\n");

    sleep(1);  // Force interleaving

    printf("Thread 2 waiting for lock1\n");
    pthread_mutex_lock(&lock1);   // DEADLOCK here

    printf("Thread 2 acquired lock1\n");

    pthread_mutex_unlock(&lock1);
    pthread_mutex_unlock(&lock2);
    return NULL;
}

int main() {
    pthread_t t1, t2;

    pthread_mutex_init(&lock1, NULL);
    pthread_mutex_init(&lock2, NULL);

    pthread_create(&t1, NULL, thread1, NULL);
    pthread_create(&t2, NULL, thread2, NULL);

    pthread_join(t1, NULL);
    pthread_join(t2, NULL);

    pthread_mutex_destroy(&lock1);
    pthread_mutex_destroy(&lock2);

    return 0;
}

Writing thread_5.c


In [ ]:
!gcc thread_5.c -o thread_5
!./thread_5

## 9. Semaphores

Semaphores generalize locks by allowing **N threads** to enter a critical region.

### Types
- Binary semaphore (mutex-like)
- Counting semaphore

### Example: Semaphore

In [30]:
%%writefile thread_6.c

#include <stdio.h>
#include <pthread.h>
#include <semaphore.h>
#include <unistd.h>

sem_t sem;

void* worker(void* arg) {
    sem_wait(&sem);
    printf("Thread %ld in critical section\n", pthread_self());
    sleep(1);
    printf("Thread %ld leaving the critical section\n", pthread_self());
    sem_post(&sem);
    return NULL;
}

int main() {
    pthread_t threads[4];
    sem_init(&sem, 0, 2);  // Allow 2 threads

    for (int i = 0; i < 4; i++)
        pthread_create(&threads[i], NULL, worker, NULL);

    for (int i = 0; i < 4; i++)
        pthread_join(threads[i], NULL);

    sem_destroy(&sem);
    return 0;
}

Overwriting thread_6.c


In [31]:
!gcc thread_6.c -o thread_6
!./thread_6

Thread 22367610320448 in critical section
Thread 22367612421696 in critical section
Thread 22367610320448 leaving the critical section
Thread 22367608219200 in critical section
Thread 22367612421696 leaving the critical section
Thread 22367606117952 in critical section
Thread 22367608219200 leaving the critical section
Thread 22367606117952 leaving the critical section


## 10. Atomic Operations

Atomic operations execute **indivisibly**, without locks.

### When to Use Atomics
- Simple shared counters
- Flags
- Lock-free data structures



In [32]:
%%writefile thread_7.c

#include <stdio.h>
#include <pthread.h>
#include <stdatomic.h>

atomic_int counter = 0;

void* increment(void* arg) {
    for (int i = 0; i < 1000000; i++) {
        atomic_fetch_add(&counter, 1);
    }
    return NULL;
}

int main() {
    pthread_t t1, t2;

    pthread_create(&t1, NULL, increment, NULL);
    pthread_create(&t2, NULL, increment, NULL);

    pthread_join(t1, NULL);
    pthread_join(t2, NULL);

    printf("Final counter: %d\n", counter);
    return 0;
}

Writing thread_7.c


In [33]:
!gcc thread_7.c -o thread_7
!./thread_7

Final counter: 2000000


## 11. Mutex vs Semaphore vs Atomic

| Feature | Mutex | Semaphore | Atomic |
|------|------|----------|--------|
| Mutual exclusion | Yes | Optional | Yes |
| Blocking | Yes | Yes | No |
| Overhead | Medium | Medium | Low |
| Complexity | Medium | High | Low |

---

## 12. Performance Considerations

- Locks serialize execution
- Fine-grained locking is better than coarse-grained
- Atomics scale better for simple updates

---

## 13. Common Mistakes

- Forgetting `pthread_join`
- Locking too much code
- Sharing stack variables
- Ignoring false sharing

---

## 14. Key Takeaways

- Threads share memory → synchronization is mandatory
- Race conditions are subtle and dangerous
- Mutexes ensure correctness
- Semaphores control access
- Atomics provide lightweight synchronization


## Exercises
For the following exercises, paste the output program below their respective questions.

1. Write a simple matrix multiplication program such that each position in the output matrix is computed by a separate thread (paste your program below)

2. Modify the program such that each position in the output matrix is now computed using multiple threads.

3. Write a program to create a deadlock situation and a livelock situation.

4. The 4th parameter in the pthreads create function accepts a reference of the variable as data. Use the thread create function inside a for-loop and pass the for loop variable as the data. The thread simply prints the number. Is the program working as expected. Why?

5. Use multiple threads to write (concatenate) 100 (random) numbers between 0 - 100 into a single shared text file, ensuring the file contents are exactly 100 numbers. Using (at least 4) threads, read the numbers from the file and build a text-based histogram showing frequency of each digit (0–9).

Tip: Explore pthreads manual to make your task easier.